# Switches

Every switch builder makes a two-terminal switch between two nodes, of the CIM class in its name:
`BreakerBuilder` makes a `Breaker`, `FuseBuilder` a `Fuse`. They all share the methods of `SwitchBuilder`:

- Built on `SwitchBuilder`: `DisconnectorBuilder`, `FuseBuilder`, `SectionaliserBuilder`,
  `GroundDisconnectorBuilder` and `ProtectedSwitchBuilder`.
- Built on `DisconnectorBuilder`: `AirBreakSwitchBuilder`, `ElbowSwitchBuilder`,
  `HookstickSwitchBuilder` and `ServiceDisconnectBuilder`.
- Built on `ProtectedSwitchBuilder`: `BreakerBuilder`, `LoadBreakSwitchBuilder` and `RecloserBuilder`.
  Their `add_electrical` adds breaking and making capacity, and a breaker's the in-transit time as well.

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs. `new_feeder_model` makes an empty model to build into.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    BaseVoltageBuilder,
    BreakerBuilder,
    FuseBuilder,
    LoadBreakSwitchBuilder,
    SwitchInfoBuilder,
    SwitchPhaseBuilder,
)

network = new_feeder_model(
    authority="https://example.com/models", model="switches", Feeder="fdr_demo"
)
container = network.container
base_voltage_builder = BaseVoltageBuilder(network, container)
for mRID, nominalVoltage in [
    ("bv_12_47kV", Qty(12.47, "kV")),
]:
    base_voltage_builder.create(mRID=mRID, name=mRID, nominalVoltage=nominalVoltage)

## A breaker

`add_connectivity` gives the switch its two Terminals and, with `phases`, one `SwitchPhase` per phase.
`normalOpen` is its normal position. `add_electrical` adds its ratings, and `add_ssh` its position in this
operating case.

In [2]:
breaker_builder = BreakerBuilder(network, container)
breaker = breaker_builder.create(mRID="brk_1", name="feeder breaker 1")
breaker, terminals, switch_phases = breaker_builder.add_connectivity(
    breaker,
    node1="subbus",
    node2="feeder_1_head",
    phases="ABC",
    normalOpen=False,
    BaseVoltage="bv_12_47kV",
)
breaker_builder.add_electrical(
    breaker,
    ratedCurrent=Qty(1200, "A"),
    breakingCapacity=Qty(25, "kA"),
    makingCapacity=Qty(65, "kA"),
    inTransitTime=Qty(50, "ms"),
)
breaker_builder.add_ssh(breaker, open=False)

print(
    [terminal.mRID for terminal in terminals], [phase.mRID for phase in switch_phases]
)
print(f"rated {breaker.ratedCurrent.to('A'):g} A")
print(f"breaking {breaker.breakingCapacity.to('kA'):g} kA")
print(f"in transit {breaker.inTransitTime.to('ms'):g} ms")

['brk_1_T1', 'brk_1_T2'] ['brk_1_A', 'brk_1_B', 'brk_1_C']
rated 1200 A
breaking 25 kA
in transit 50 ms


## A normally open tie

A tie between two feeders is normally open. Set `normalOpen=True` on the switch and `open=True` in its
operating state, unless this case has it closed.

In [3]:
tie_builder = LoadBreakSwitchBuilder(network, container)
tie = tie_builder.create(mRID="tie_1_2", name="tie 1-2")
tie_builder.add_connectivity(
    tie,
    node1="feeder_1_end",
    node2="feeder_2_end",
    phases="ABC",
    normalOpen=True,
    BaseVoltage="bv_12_47kV",
)
tie_builder.add_ssh(tie, open=True)
print(type(tie).__name__, tie.normalOpen, tie.open)

LoadBreakSwitch True True


## One phase open

Each phase of a switch can have its own position. Here phase B of a fuse has blown. The `SwitchPhase`
objects made by `add_connectivity` are named `<switch>_<phase>`, and `SwitchPhaseBuilder.add_ssh` sets
the position of each one.

In [4]:
fuse_builder = FuseBuilder(network, container)
fuse = fuse_builder.create(mRID="fuse_684", name="684")
fuse_builder.add_connectivity(
    fuse, node1="671", node2="684", phases="ABC", BaseVoltage="bv_12_47kV"
)
fuse_builder.add_electrical(fuse, ratedCurrent=Qty(100, "A"))

phase_builder = SwitchPhaseBuilder(network, container)
for phase in "ABC":
    phase_builder.add_ssh(
        network.find_by_mRID(f"fuse_684_{phase}"), open=phase == "B"
    )

print([(phase.mRID, phase.open) for phase in fuse.SwitchPhase])

[('fuse_684_A', False), ('fuse_684_B', True), ('fuse_684_C', False)]


`SwitchPhaseBuilder.add_connectivity` makes a `SwitchPhase` by hand instead, with `phaseSide1` and
`phaseSide2` for a switch that connects different phases on its two sides.

## Switch catalog data

Ratings shared by every switch of one type can go on a `SwitchInfo` catalog entry instead. Any piece of
equipment links to its catalog entry with `add_asset_datasheet`:

In [5]:
info_builder = SwitchInfoBuilder(network, container)
info = info_builder.create(mRID="swinfo_vcb_15kV", name="15 kV vacuum breaker")
info_builder.add_asset_info(
    info,
    ratedVoltage=Qty(15, "kV"),
    ratedCurrent=Qty(1200, "A"),
    breakingCapacity=Qty(25, "kA"),
    ratedInterruptingTime=Qty(50, "ms"),
)

breaker_builder.add_asset_datasheet(breaker, AssetDatasheet="swinfo_vcb_15kV")
print(
    breaker.AssetDatasheet.name,
    [equipment.mRID for equipment in info.PowerSystemResources],
)

15 kV vacuum breaker ['brk_1']
